# 03 · Training and Simulated Evaluation

This notebook fits the four fixed benchmark models on the 24 training examples and classifies the 12 held-out test examples once, with the quantum-kernel SVM using the exact statevector-simulated kernel. It runs the project's own scripts (`scripts/train_benchmark.py`, then `scripts/evaluate_benchmark.py`), checks that the recomputed evaluation equals the published results, and reproduces the training-only leave-one-out diagnostic recorded in the protocol. Nothing is tuned: the model settings were fixed before any test result was seen.

**Inputs**
- `configs/benchmark-split.json`, `configs/feature-scaler.json`
- `data/processed/benchmark-sample.csv` (produced by the earlier notebooks)
- `reports/benchmark-results.json`, `reports/benchmark-replay.json` (published files, used only for comparison; never overwritten)
- `docs/benchmark-protocol.md` (recorded numbers)

**Outputs**
- `artifacts/benchmark/models.pkl`, `training-angles.npy`, `training-kernel.npy`, `training-summary.json`
- `artifacts/benchmark/evaluation-simulation.json`, `test-kernel-ideal.npy`

Run after: notebook 02 (features and angle scaling).

In [ ]:
import os
import runpy
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir() and (p / "configs").is_dir())
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

import json
import math

import numpy as np
import pandas as pd
from sklearn.model_selection import LeaveOneOut, cross_val_score

from src.evaluation import inputs as benchmark_inputs
from src.evaluation import publish
from src.evaluation.metrics import METRIC_NAMES, evaluate
from src.models import benchmark_models
from src.quantum import kernel as qkernel

ARTIFACTS = Path("artifacts/benchmark")
print("Repository root located; working directory set to it.")

## 1. Fixed model settings

The protocol (section "Fixed model configurations") freezes four models before any test evaluation: no hyperparameter search, no cross-validation model selection, no class weights, no threshold tuning. `src.models.benchmark_models.settings` reports the parameters that matter; `fit_all` treats a `ConvergenceWarning` as an error, so a non-converged logistic regression stops the run instead of being published.

In [ ]:
fixed = benchmark_models.settings(benchmark_models.build_models())
pd.DataFrame(fixed).T.astype(object).fillna("")

In [ ]:
# Replication check: the settings are exactly the ones written in the protocol.
expected_settings = {
    "majority": {"strategy": "most_frequent"},
    "logistic_regression": {"C": 1.0, "class_weight": None, "max_iter": 1000, "solver": "lbfgs"},
    "rbf_svm": {"C": 1.0, "class_weight": None, "gamma": "scale", "kernel": "rbf", "max_iter": -1},
    # gamma is unused with a precomputed kernel; max_iter=-1 is the SVC default (no iteration limit).
    "quantum_kernel_svm": {"C": 1.0, "class_weight": None, "gamma": "scale", "kernel": "precomputed", "max_iter": -1},
}
assert fixed == expected_settings, fixed
assert tuple(fixed) == benchmark_models.MODEL_NAMES
print("OK: four fixed model settings match the protocol (no class weights, default decision rules).")

## 2. Train (no test data)

`scripts/train_benchmark.py` reads only the training partition, scales it with the frozen scaler, builds the 24 x 24 exact statevector kernel of the ZZ feature map (4 qubits, reps=1, linear entanglement, alpha=2), checks it is a valid kernel, and fits the four models. Its summary records `uses_test_data: false`.

In [ ]:
train_script = runpy.run_path("scripts/train_benchmark.py", run_name="train_benchmark")
rc = train_script["main"]()
assert rc == 0, rc
print("train_benchmark.py return code:", rc)

In [ ]:
# Replication check: the training run used no test data, matches the frozen inputs, and passed the kernel checks.
summary = json.loads((ARTIFACTS / "training-summary.json").read_text(encoding="utf-8"))
hashes = {
    "split_manifest_sha256": benchmark_inputs.sha256(benchmark_inputs.MANIFEST_PATH),
    "scaler_sha256": benchmark_inputs.sha256(benchmark_inputs.SCALER_PATH),
}
assert summary["uses_test_data"] is False
assert summary["inputs"] == hashes
assert summary["training_labels"] == {"legitimate": 12, "phishing": 12}
assert summary["feature_map"] == qkernel.FEATURE_MAP_SETTINGS
assert summary["models"] == expected_settings
assert summary["kernel_diagnostics"]["eigenvalue_check_passed"] is True
# The split on disk must be the one the published results came from: the published replay file names
# the simulation run "sim-" + the first 8 hex digits of the split manifest's sha256.
published_sim_run = json.loads(Path("reports/benchmark-replay.json").read_text(encoding="utf-8"))["source_runs"]["simulation"]
assert published_sim_run == "sim-" + hashes["split_manifest_sha256"][:8], (
    f"split manifest changed: published run {published_sim_run}, local sha256 prefix {hashes['split_manifest_sha256'][:8]}"
)
print("OK: training used 24 rows (12/12), no test data, frozen split/scaler hashes, valid kernel.")
print(f"OK: split manifest sha256 prefix {hashes['split_manifest_sha256'][:8]} matches the published run {published_sim_run}.")

## 3. Classify the test set once (exact simulation)

`scripts/evaluate_benchmark.py` loads the frozen inputs and fitted models (refusing any mismatch), simulates the 12 x 24 test kernel exactly, and evaluates all four models with their default decision rules. Because `reports/benchmark-results.json` is already complete (it contains the IBM hardware rows), the script keeps that file untouched and writes its evaluation to `artifacts/benchmark/evaluation-simulation.json`.

In [ ]:
published_before = Path("reports/benchmark-results.json").read_bytes()
eval_script = runpy.run_path("scripts/evaluate_benchmark.py", run_name="evaluate_benchmark")
rc = eval_script["main"]()
assert rc == 0, rc
assert Path("reports/benchmark-results.json").read_bytes() == published_before
print("evaluate_benchmark.py return code:", rc)
print("OK: the published results file was left unchanged.")

## 4. Replication checks against the published results

One consistency check and two replication checks against the published files:
1. Consistency (not independent evidence): the script's `evaluation-simulation.json` against an evaluation recomputed here with the same functions on the same frozen inputs (predictions and confusion matrices exact, metrics and decision scores within 1e-9). It only shows that the script and this notebook agree.
2. Replication: every classical row and the simulation quantum-kernel row in `reports/benchmark-results.json` against the recomputation.
3. Replication: the quantum-kernel SVM's per-sample simulation predictions and margins in `reports/benchmark-replay.json`.

In [ ]:
data = benchmark_inputs.load()
test_kernel = qkernel.statevector_kernel(data.test_angles, data.train_angles)
assert test_kernel.shape == (12, 24)
recomputed = {
    name: evaluate(model, test_kernel if name == "quantum_kernel_svm" else data.test_angles, data.test_labels)
    for name, model in data.models.items()
}
detail = json.loads((ARTIFACTS / "evaluation-simulation.json").read_text(encoding="utf-8"))
published = json.loads(Path("reports/benchmark-results.json").read_text(encoding="utf-8"))
replay = json.loads(Path("reports/benchmark-replay.json").read_text(encoding="utf-8"))
print("Test labels (legitimate / phishing):", int((data.test_labels == 0).sum()), "/", int(data.test_labels.sum()))
print("Values clipped by the scaler on test rows:", data.test_clipped)

In [ ]:
def close(a, b, tol=1e-9):
    if a is None or b is None:
        return a is None and b is None
    return math.isclose(a, b, rel_tol=0, abs_tol=tol)

# Check 1 (artifact consistency, not replication): the script's artifact equals the in-notebook recomputation.
assert np.allclose(np.load(ARTIFACTS / "test-kernel-ideal.npy"), test_kernel, rtol=0, atol=1e-12)
assert detail["inputs"] == data.input_hashes
for name, ours in recomputed.items():
    theirs = detail["evaluations"][name]
    assert theirs["predictions"] == ours["predictions"], name
    assert theirs["confusion_matrix"] == ours["confusion_matrix"], name
    assert all(close(theirs["metrics"][m], ours["metrics"][m]) for m in METRIC_NAMES), name
    if ours["decision_scores"] is None:
        assert theirs["decision_scores"] is None, name
    else:
        assert np.allclose(theirs["decision_scores"], ours["decision_scores"], rtol=0, atol=1e-9), name
print("OK (consistency): evaluation-simulation.json equals the in-notebook recomputation (predictions exact).")

In [ ]:
# Check 2: classical rows and the simulation quantum row of the published results file.
rows = {(r["name"], r["kernel_source"]): r for r in published["models"]}
for name, (display, source) in publish.DISPLAY.items():
    row = rows[(display, source)]
    ours = recomputed[name]
    assert row["confusion_matrix"] == ours["confusion_matrix"], (display, row["confusion_matrix"])
    for metric in METRIC_NAMES:
        assert close(row["metrics"][metric], ours["metrics"][metric]), (display, metric)
    print(f"OK: {display} [{source}] metrics within 1e-9, confusion matrix exact {ours['confusion_matrix']}.")
assert published["split"] == {"train": 24, "test": 12, "seed": 42}

In [ ]:
# Check 3: per-sample simulation predictions and margins in the published replay file.
manifest = json.loads(Path("configs/benchmark-split.json").read_text(encoding="utf-8"))
expected_run_id = "sim-" + data.input_hashes["split_manifest_sha256"][:8]
assert replay["source_runs"]["simulation"] == expected_run_id == detail["run_id"]
qsvm = recomputed["quantum_kernel_svm"]
assert len(replay["samples"]) == 12
for sample, ref, pred, score, label in zip(
    replay["samples"], manifest["test"], qsvm["predictions"], qsvm["decision_scores"], data.test_labels, strict=True
):
    assert sample["row_index"] == ref["row_index"] and sample["is_phishing"] == ref["is_phishing"] == int(label)
    assert sample["position"] == ref["position"] and sample["group_hash"] == ref["group_hash"]
    sim = sample["predictions"]["simulation"]
    assert sim["is_phishing"] == pred
    assert close(sim["decision_score"], score)
print(f"OK: replay file ({expected_run_id}) matches all 12 test entries of the split (position, row_index, group_hash, label), simulation predictions exactly and margins within 1e-9.")

## 5. Results table

Accuracy is also shown as a count out of 12: one test example moves accuracy by about 8.3 percentage points. Average precision is computed from continuous decision scores; the majority baseline has none, so it is not available. SVM margins are not probabilities.

In [ ]:
SOURCE_LABEL = {"classical": "classical", "simulation": "exact simulation"}
table = []
for name, ev in recomputed.items():
    display, source = publish.DISPLAY[name]
    (tn, fp), (fn, tp) = ev["confusion_matrix"]
    m = ev["metrics"]
    table.append({
        "model": display,
        "source": SOURCE_LABEL[source],
        "correct / 12": f"{tn + tp} / 12",
        **{k: ("n/a" if m[k] is None else round(m[k], 3)) for k in METRIC_NAMES},
        "TN / FP / FN / TP": f"{tn} / {fp} / {fn} / {tp}",
    })
results_table = pd.DataFrame(table).set_index("model")
results_table

## 6. Training-only diagnostic: leave-one-out on the 24 training examples

The protocol records a diagnostic run after the test evaluation that changes nothing: leave-one-out accuracy on the training examples only (documented: 0.54 for the quantum-kernel SVM, 0.50 for the RBF SVM), and the class structure of the training kernel (mean similarity 0.245 within a class, 0.199 across classes; the quantum-kernel SVM fits 83% of its training examples). It uses the same fixed settings: the quantum-kernel SVM receives the precomputed training kernel (scikit-learn slices rows and columns per fold), the classical models receive the training angles. No test data is used, and nothing is selected from it.

Logistic regression and the majority baseline are shown for completeness; the protocol records only the two SVM values. The majority baseline scores 0/24 by construction: removing one example leaves the other class in the majority, so it always predicts the wrong class.

In [ ]:
train_kernel = np.load(ARTIFACTS / "training-kernel.npy")
y_train = data.train_labels
assert np.allclose(train_kernel, qkernel.statevector_kernel(data.train_angles), rtol=0, atol=1e-12)

loo = {}
for name, model in benchmark_models.build_models().items():
    features = train_kernel if name == "quantum_kernel_svm" else data.train_angles
    scores = cross_val_score(model, features, y_train, cv=LeaveOneOut())
    loo[name] = (int(scores.sum()), float(scores.mean()))

off_diagonal = ~np.eye(len(y_train), dtype=bool)
same_class = y_train[:, None] == y_train[None, :]
within = float(train_kernel[off_diagonal & same_class].mean())
across = float(train_kernel[~same_class].mean())
train_fit = float((data.models["quantum_kernel_svm"].predict(train_kernel) == y_train).mean())

pd.DataFrame(
    {"LOO correct / 24": [c for c, _ in loo.values()], "LOO accuracy": [round(a, 3) for _, a in loo.values()]},
    index=[publish.DISPLAY[n][0] + " (training data only)" for n in loo],
)

In [ ]:
# Replication check: the documented training-only diagnostic, to 2 decimals (3 for kernel similarities).
assert round(loo["quantum_kernel_svm"][1], 2) == 0.54, loo["quantum_kernel_svm"]
assert round(loo["rbf_svm"][1], 2) == 0.50, loo["rbf_svm"]
assert round(within, 3) == 0.245 and round(across, 3) == 0.199, (within, across)
assert round(train_fit, 2) == 0.83, train_fit
print(f"OK: LOO accuracy quantum-kernel SVM {loo['quantum_kernel_svm'][1]:.2f}, RBF SVM {loo['rbf_svm'][1]:.2f} (protocol: 0.54, 0.50).")
print(f"OK: training kernel mean similarity within class {within:.3f}, across classes {across:.3f} (protocol: 0.245, 0.199).")
print(f"OK: quantum-kernel SVM fits {train_fit:.0%} of its training examples (protocol: 83%).")

## 7. Interpretation

- **Quantum-kernel SVM (exact simulation):** 6 of 12 test examples correct, balanced accuracy 0.500, the same as the majority baseline. On this split it performs at chance. Its training kernel separates the classes only weakly (0.245 vs 0.199 mean similarity), so it can fit most training examples (83%) without generalizing (leave-one-out 0.54). No quantum advantage is claimed or suggested by these numbers.
- **RBF SVM (classical):** 11 of 12 correct on the test set, but only 0.50 leave-one-out accuracy on the training data, no better than chance. With 24 training and 12 test examples, one test example moves accuracy by 8.3 points, and the test score depends on which domains happened to land in the test set. The 0.917 is therefore an unstable estimate, not evidence that the RBF SVM is a strong phishing detector on these features.
- **Logistic regression (classical):** 10 of 12 correct; on this split it never flags a legitimate host (precision 1.0).
- **Majority baseline (classical):** breaks the 12/12 training tie toward legitimate, so it never predicts phishing; it has no decision scores, so average precision is not available.
- **Scope:** one domain-disjoint split, one seed (42), four host-only features; no variability estimate across splits or seeds exists. The diagnostic in section 6 was run after the test evaluation and changes no setting; nothing in this notebook is tuned.

The IBM hardware rows (`ibm_fez`, uncorrected and calibrated) reuse these same fitted models and only replace the test kernel; they are reproduced from the versioned recorded counts in a later notebook.